# What Frequency Cannot Show

**Computational Text Analysis Pack · Module 11 — From Counts to Argument**

*Technique:* The limits section; silence, negation, and who is absent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/11b_what_frequency_cannot_show.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The limits section is where you say what your evidence cannot do. Done properly it
makes a paper stronger, because it shows you know the shape of your own method.

Five things frequency analysis cannot see. Each is demonstrated here, not
asserted — so you can cite the demonstration rather than the intuition.

## The tooling

No new library. This module is about turning numbers into an argument, which is
the part no package does for you.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

## 1. Negation

In [ ]:
import re
# The colonial corpus is the sharper demonstration: it contains both
# pro-slavery and abolitionist writing, and counting cannot tell them apart.
TERM = "slavery"            # ← change this
source = df_colonial

pat = re.compile(rf"\b{TERM}\b", re.I)
neg_pat = re.compile(r"\b(no|not|never|denied|deny|against|anti|fight\w*|combat\w*|"
                     r"tackl\w*|end)\b", re.I)
plain, negated = 0, 0
for t in source["text"]:
    s = str(t)
    for m in pat.finditer(s):
        window = s[max(0, m.start() - 60):m.start()]
        if neg_pat.search(window):
            negated += 1
        else:
            plain += 1
print(f"'{TERM}': {plain + negated} occurrences")
print(f"   {negated} ({negated / max(plain + negated, 1):.0%}) preceded by a negation")
print("   or opposition word within 60 characters")
print()
print("A frequency count treats all of these identically. 'The abolition of")
print("slavery' and 'the defence of slavery' are the same token to every method")
print("in this pack — and this corpus contains both.")

## 2. Who is speaking

In [ ]:
eng = df[df["language"] == "English"]
QUOTE_TERM = "government"   # ← change this
qpat = re.compile(rf"\b{QUOTE_TERM}\b", re.I)

quoted = 0
total = 0
for t in eng["text"]:
    s = str(t)
    for m in qpat.finditer(s):
        total += 1
        before = s[:m.start()]
        # Inside a quotation if an odd number of quote marks precedes it.
        if (before.count('"') + before.count('\u201c') + before.count('\u201d')) % 2 == 1:
            quoted += 1
if total:
    print(f"'{QUOTE_TERM}': {quoted} of {total} occurrences "
          f"({quoted / total:.0%}) fall inside quotation marks")
print()
print("A word inside a quotation is the source's word, not the outlet's. Every")
print("method in this pack attributes it to the document.")

## 3. What is absent

In [ ]:
# The absence audit, for text. What is NOT in this corpus?
print("MasakhaNEWS: documents per language")
print(df["language"].value_counts().to_string())
print()
langs_in_africa = 2000      # conservative; Glottolog lists far more
print(f"languages represented   : {df['language'].nunique()}")
print(f"languages spoken in Africa: ~{langs_in_africa:,}")
print(f"coverage                : {df['language'].nunique() / langs_in_africa:.2%}")
print()
print("Every language without a digital news industry is invisible here, and")
print("that absence correlates with exactly the things a researcher cares about:")
print("colonial history, state investment, script availability, speaker numbers.")

In [ ]:
# The same audit on the colonial corpus, which you did in 01b.
print(f"colonial corpus: {df_colonial['work_id'].nunique()} works, "
      f"{df_colonial['author'].nunique()} authors")
print()
print("Absent by construction:")
for line in ["everything published after 1929 (still in copyright)",
             "everything not in English",
             "everything oral — which is most African literature before 1929",
             "everything never digitised by Project Gutenberg's volunteers"]:
    print(f"  · {line}")
print()
print("A finding about 'what was written about Africa' is really a finding about")
print("what survived, was printed, was in English, and was later scanned.")

## 4. Change over time

In [ ]:
print("columns available in the news corpus:")
print("  " + ", ".join(df.columns))
print()
print("There is no date column. So no trend, no 'increasingly', no 'since'.")
print()
print("This is worth saying out loud because temporal language slips into")
print("write-ups by habit. If your corpus has no dates, your findings have no")
print("direction — and a reviewer will notice before you do.")

## 5. Meaning

In [ ]:
from collections import Counter
counts = Counter()
for t in eng["text"]:
    counts.update(tokenise(t))

for w in ("state", "party", "bank", "left", "charge"):
    if counts[w]:
        print(f"'{w}': {counts[w]:,} occurrences — and at least two distinct senses")
print()
print("No method here disambiguates senses. 'State' as government and 'state'")
print("as a region are one token. In a Nigerian corpus that particular collapse")
print("is severe, because both senses are constantly in play.")

### 🔧 Your turn 1

Run the negation check on a word central to your own research.

If more than about a fifth of occurrences are negated or oppositional, a raw
frequency claim about that word is not defensible without reading. Say so — and
then read.

## Writing the limits section

Four sentences, in this order. It fits in a paragraph and it is worth as much as
any result you report.

1. **What the corpus is** — how it was assembled, and by what filter
2. **Who or what is absent**, and whether the absence is systematic
3. **What the method cannot see** — pick the two from this notebook that bite
4. **The strongest alternative explanation** you cannot rule out

In [ ]:
template = """
LIMITS

This analysis uses [CORPUS], assembled by [HOW], comprising [N] documents in
[WHAT]. It excludes [WHAT IS ABSENT], an omission that is [random / systematic]
because [WHY].

The methods used count word forms and cannot distinguish [NEGATION / QUOTATION /
SENSE / IRONY]. In this corpus, [N]% of occurrences of [KEY TERM] are [NEGATED /
QUOTED], so the reported rate overstates [WHAT].

The strongest alternative explanation for the pattern reported is [ALTERNATIVE],
which the present data cannot rule out because [WHY]. Testing it would require
[WHAT DATA].
"""
print(template)

### 🔧 Your turn 2

Fill in that template for the analysis you are building towards in Module 12.

Do it **now**, before the analysis is finished. Written first, a limits section
tells you which results are worth pursuing; written last, it is an apology.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Words naming a problem — *corruption*, *violence*, *poverty*,
*insecurity* — are heavily negated or oppositional in news text, often above 30%,
because journalism reports on efforts against them. A frequency claim about such
a word without this check will be wrong in direction, not just in degree. The
regex here is crude and will miss distant negation; treat its number as a floor.

**Your turn 2.** The template is deliberately mechanical, and filling it in early
frequently kills a planned analysis — which is the cheapest possible time for that
to happen. If you cannot complete sentence 2, you do not yet know your corpus. If
you cannot complete sentence 4, you have not yet thought of the objection a
reviewer will make first.

</details>

## Where this stops

You know what your evidence can and cannot carry. Next: **Module 12 — the
capstone**, where you do the whole thing on a corpus of your own choosing.